<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download-url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>
<br clear="all"/>

---

# Track 1 (Notebook 02): Agentic Medallion Transformation (`acsm_bronze` → `acsm_silver` → `acsm_gold` + Airflow Orchestration)
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

> 📚 **Reference Guide (Learn More)**: [Building a Medallion Architecture in BigQuery using Data Engineering Agents (`discuss.google.dev`)](https://discuss.google.dev/t/building-a-medallion-architecture-in-bigquery-using-data-engineering-agents/281782)

---

## 🗺️ Notebook 02 Architecture & Medallion DAG (`acsm_bronze` → `acsm_silver` → `acsm_gold`)

![Track 1 Notebook 02 — Agentic Medallion Transformation Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook2_medallion_architecture_flow.png)

---
### 📋 Notebook 02 Step-by-Step Execution Summary (100% Standalone — No Need to Run Notebook 01 First!)
0. **Step 0**: Configure parameterized `PROJECT_ID` (auto-detects active project if blank) & `LOCATION = "asia-southeast1"` (Singapore), enable the Data Engineering Agent APIs (`dataform.googleapis.com`, `cloudaicompanion.googleapis.com`), and stage all 8 compressed `.csv.gz` files in `gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/`.
1. **Step 1 (Bootstrap `acsm_bronze` Layer via `.sql` Scripts)**: Invoke `00_create_8_tables_ddl_with_descriptions.sql` and `01_load_data_from_gcs.sql` to create and load all **8 `acsm_bronze` tables** as native BigQuery `BASE TABLE`s so this notebook can be executed independently without running Notebook 01 first.
2. **Step 2 (`%%bigquery` SQL)**: Verify row counts across all **8 `acsm_bronze` base tables** (`1,398,284` rows).
3. **Step 3 (`%%bigquery` SQL)**: Create the target Medallion & Dataform datasets (`acsm_silver`, `acsm_gold`, `dataform`, and `dataform_assertions`) in `asia-southeast1`.
4. **Step 4 (UI Guide + Consolidated Natural-English Prompts)**: Copy-paste the **Phase 1 & Phase 2 Consolidated Natural-English Prompts** (Medallion DAG + Incremental Load + Assertions $\rightarrow$ BQML Model Training `acsm_silver.model_delinquency_propensity` + Batch BQML Scoring `acsm_gold.gold_aeon360_batch_ml_predictions`) into the **BigQuery Data Engineering Agent (`+ Create` → `Pipeline`)**.
5. **Step 5 (Optional SQL Fallback — Invoke `02_medallion_and_reconciliation.sql`)**: Optional fallback step that invokes [`02_medallion_and_reconciliation.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/02_medallion_and_reconciliation.sql) to build the Silver/Gold Medallion tables, BQML model, and batch predictions directly in case you skip running the Dataform Pipeline in Step 4.
6. **Step 6 (`%%bigquery` SQL)**: Verify `acsm_silver` & `acsm_gold` row counts, run the **Dual-Run Financial Control Total Reconciliation Audit (`0.00 MYR Variance`)**, and preview the batch ML predictions.
7. **Step 7 (Apache Airflow / Cloud Composer Orchestrator)**: Invoke [`invoke_dataform_medallion_dag.py`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/airflow/invoke_dataform_medallion_dag.py) to programmatically compile and execute the Dataform Medallion DAG in `asia-southeast1`.

---
## Step 0: Configure Project & Region (`asia-southeast1`), Enable Data Engineering Agent APIs & Stage `.csv.gz` Files
Run the cell below to set your `PROJECT_ID` and `LOCATION` (`asia-southeast1` Singapore), load the `%bigquery` SQL magic extension, enable the required APIs (`dataform.googleapis.com` and `cloudaicompanion.googleapis.com`), and stage all 8 compressed `.csv.gz` files into `gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/`.

In [ ]:
# @title 0.1 Set Parameterized `PROJECT_ID` (Auto-Detects Active Project if Blank), Singapore Region (`asia-southeast1`), Enable APIs & Stage All 8 `.csv.gz` Files in GCS
import os
import subprocess

PROJECT_ID = ""  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID.startswith("<"):
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
    )
LOCATION = "asia-southeast1"  # @param {type:"string"}
BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

# Load the native BigQuery SQL magic so all subsequent cells run pure SQL against $PROJECT_ID
%load_ext google.cloud.bigquery

# 1. Enable BigQuery Pipelines (Dataform), BigLake, Cloud Storage, and Gemini for Google Cloud (Data Engineering Agent)
!gcloud config set project $PROJECT_ID
!gcloud services enable storage.googleapis.com bigquery.googleapis.com biglake.googleapis.com dataform.googleapis.com cloudaicompanion.googleapis.com --project=$PROJECT_ID

# 2. Ensure all 8 compressed .csv.gz files are staged in gs://acsm-workshop-landing-${PROJECT_ID}/full_compressed/
#    so Notebook 02 can run 100% independently even if Notebook 01 was not run first!
![ -d aeon-credit-gcp-workshop ] || git clone https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

# 3. Provision service identities and grant BigQuery + Storage permissions to the Dataform & Gemini Service Agents
PROJECT_NUMBER = subprocess.check_output(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"], text=True
).strip()

!gcloud beta services identity create --service=dataform.googleapis.com --project=$PROJECT_ID 2>/dev/null || true
!gcloud beta services identity create --service=cloudaicompanion.googleapis.com --project=$PROJECT_ID 2>/dev/null || true

for sa in [
    f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-dataform.iam.gserviceaccount.com",
    f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-cloudaicompanion.iam.gserviceaccount.com",
    f"serviceAccount:service-{PROJECT_NUMBER}@gcp-sa-bigqueryconnection.iam.gserviceaccount.com",
]:
    for role in [
        "roles/biglake.admin",
        "roles/storage.objectAdmin",
        "roles/bigquery.connectionUser",
        "roles/bigquery.dataEditor",
        "roles/bigquery.jobUser",
    ]:
        subprocess.run(
            ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID, f"--member={sa}", f"--role={role}", "--quiet"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )

print(f"✅ Ready! Active Project={PROJECT_ID} (#{PROJECT_NUMBER}) | Region={LOCATION} (Singapore)")
print(f"✅ All 8 compressed .csv.gz files staged in gs://{BUCKET_NAME}/full_compressed/ for standalone execution!")

---
## Step 1 (Standalone Bronze Layer Bootstrap): Invoke `.sql` Scripts to Create & Load All 8 `acsm_bronze` Base Tables
Run the cell below to execute `00_create_8_tables_ddl_with_descriptions.sql` (DDL) and `01_load_data_from_gcs.sql` (DML) so all **8 `acsm_bronze` tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection`, `m3CIF`, and `dimProduct`) exist as native BigQuery `BASE TABLE`s in `asia-southeast1` — even if Notebook 01 was not run first.

In [ ]:
# @title 1.1 Invoke `00_create_8_tables_ddl_with_descriptions.sql` & `01_load_data_from_gcs.sql` to Bootstrap `acsm_bronze`
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_data_from_gcs.sql
print("✅ Bronze layer (`acsm_bronze`) initialized with all 8 base tables!")

---
## Step 2: Pre-Flight Check — Verify the 8 Source Tables in `acsm_bronze` (`%%bigquery`)
Before creating the Silver and Gold Medallion layers, run the `%%bigquery` SQL cell below to verify that all **8 core ACSM tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection`, `m3CIF`, `dimProduct`) are loaded in `acsm_bronze`.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Verify all 8 source tables & views in acsm_bronze (6 Native Fact Tables + m3CIF Lakehouse Iceberg View + dimProduct AWS Glue Federated Iceberg View)
SELECT 'Fact_EP_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Judge`
UNION ALL
SELECT 'Fact_EP_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Sales`
UNION ALL
SELECT 'Fact_EP_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Collection`
UNION ALL
SELECT 'Fact_CC_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Judge`
UNION ALL
SELECT 'Fact_CC_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Sales`
UNION ALL
SELECT 'Fact_CC_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Collection`
UNION ALL
SELECT 'm3CIF'              AS table_name, 'GCP Lakehouse Iceberg View (GCS)'     AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.m3CIF`
UNION ALL
SELECT 'dimProduct'         AS table_name, 'AWS Glue Federated Iceberg View (S3)' AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.dimProduct`
ORDER BY table_name;

---
## Step 3: Create the Target Medallion & Dataform Datasets (`acsm_silver`, `acsm_gold`, `dataform`, `dataform_assertions`) in Singapore (`asia-southeast1`)
The **BigQuery Data Engineering Agent** requires the target Medallion datasets (`acsm_silver` and `acsm_gold`) as well as the default Dataform working and assertion datasets (`dataform` and `dataform_assertions` referenced in `workflow_settings.yaml`) to exist in the same region (`asia-southeast1` Singapore) as `acsm_bronze` before generating and running the Dataform Medallion pipeline.

> 💡 **Note on BQML Model Creation**: Both the **BigQuery ML Delinquency Propensity Model (`acsm_silver.model_delinquency_propensity`)** and the downstream **Gold Batch Prediction Table (`acsm_gold.gold_aeon360_batch_ml_predictions`)** are created directly through **natural-language prompts in the Data Engineering Agent in Step 4** (and included in the optional fallback SQL in **Step 5**).

Run the `%%bigquery` SQL cell below to create all required datasets (`acsm_silver`, `acsm_gold`, `dataform`, and `dataform_assertions`) in `asia-southeast1` (~2 seconds).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Create the Silver Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_silver`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Silver Medallion Layer: Standardized, type-cast, and deduplicated Customer CIF, EP/CC Underwriting, Collections tables, and BQML Delinquency Propensity Model in Singapore (asia-southeast1)'
);

-- 2. Create the Gold Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_gold`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Gold Medallion Layer: Unified AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store and Batch ML Predictions in Singapore (asia-southeast1)'
);

-- 3. Create Default Dataform & Dataform Assertions Datasets in Singapore (asia-southeast1)
-- Required by workflow_settings.yaml (defaultDataset: dataform, defaultAssertionDataset: dataform_assertions)
CREATE SCHEMA IF NOT EXISTS `dataform`
OPTIONS (
  location = 'asia-southeast1',
  description = 'Default Dataform working dataset in Singapore (asia-southeast1)'
);

CREATE SCHEMA IF NOT EXISTS `dataform_assertions`
OPTIONS (
  location = 'asia-southeast1',
  description = 'Default Dataform assertion views & test results dataset in Singapore (asia-southeast1)'
);

-- 4. Verify all Medallion & Dataform Datasets in INFORMATION_SCHEMA
SELECT
  schema_name AS dataset_name,
  location,
  creation_time
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name IN ('acsm_bronze', 'acsm_silver', 'acsm_gold', 'dataform', 'dataform_assertions')
ORDER BY schema_name;

---
## Step 4: How & Where to Copy-Paste the Medallion & BQML Prompts in the BigQuery Data Engineering Agent

> 📖 **Learn More — Official Reference Guide**: [Building a Medallion Architecture in BigQuery using Data Engineering Agents (`https://discuss.google.dev/t/building-a-medallion-architecture-in-bigquery-using-data-engineering-agents/281782`)](https://discuss.google.dev/t/building-a-medallion-architecture-in-bigquery-using-data-engineering-agents/281782)

Now that `acsm_bronze` (with **all 8 base tables created and loaded in Step 1**), `acsm_silver`, `acsm_gold`, `dataform`, and `dataform_assertions` are ready in **`asia-southeast1` (Singapore)**, follow these clicks in **BigQuery Studio** to generate your **Medallion + Incremental Load + Dataform Assertions + BQML Model Training + Batch ML Scoring + SLA Alerting Pipeline DAG** using plain natural English.

### 🧭 Part A: Where to Click in BigQuery Studio
1. Open **Google Cloud Console** $\rightarrow$ navigate to **BigQuery** $\rightarrow$ **Studio**.
2. In the left **Explorer** pane, expand your project (`${PROJECT_ID}`) and verify you see the datasets:
   - `acsm_bronze` *(contains all 8 source base tables enriched with 100% `Mock Metadata.xlsx` table & column descriptions)*
   - `acsm_silver` & `acsm_gold` *(created in Step 3)*
   - `dataform` & `dataform_assertions` *(created in Step 3 for `workflow_settings.yaml`)*
3. At the top of the **BigQuery Studio workspace tab bar** (next to **`+ SQL query`** and **`+ Notebook`**), click **`+` (Create new)** $\rightarrow$ select **`Pipeline`**.
4. When prompted for the **Pipeline Location / Code Region**, select **`asia-southeast1 (Singapore)`**.
5. Inside the **Pipeline Canvas**, click the **Gemini Sparkle button (`Ask Data Engineering Agent` / `Generate with Gemini`)**.

---
### 📋 Part B: 2-Phase Consolidated Natural-English Prompts
Because BigQuery's dry-run validator requires `acsm_silver.model_delinquency_propensity` to exist in BigQuery before validating a downstream `ML.PREDICT` query, run **Phase 1 (Build Medallion + Train BQML Model)** $\rightarrow$ click **`Run`**, and then paste **Phase 2 (Batch BQML Delinquency Propensity Prediction + SLA Schedule)**:

#### **Phase 1 Consolidated Prompt — Build 3-Tier Medallion Tables, Incremental Logic, Dataform Assertions (`workflow_settings.yaml`) & Train BQML Delinquency Model (`acsm_silver.model_delinquency_propensity`)**
```text
Using the source tables in the `acsm_bronze` dataset, build a 3-tier Medallion pipeline and train a BigQuery ML delinquency model in `acsm_silver` and `acsm_gold`:

1. In `acsm_silver`, create four cleansed domain tables:
   - `silver_customer_cif`: Deduplicate customer master records from `acsm_bronze.m3CIF` by customer ID (`CIF_ID`), keeping the most recent record based on `Rcd_DT`, and retain `Rcd_DT` along with standardized demographic and income fields (`N_Age`, `B_NetIncome`, `B_AnnualIncome`, `State`, `Region`, `Occupation`).
   - `silver_ep_underwriting`: Cleanse Easy Payment loan applications from `acsm_bronze.Fact_EP_Judge`, standardize the customer ID, retain `APPL_DT` as the application date column, and retain key underwriting score, DSR, NDI, and financing amount metrics.
   - `silver_cc_underwriting`: Cleanse Credit Card applications from `acsm_bronze.Fact_CC_Judge`, retain `Appl_DT` as the application date column, and retain credit score, DSR, NDI, and approved credit limit metrics.
   - `silver_collections_summary`: Combine Easy Payment (`Fact_EP_Collection`) and Credit Card (`Fact_CC_Collection`) collections at the customer level to calculate total unpaid principal and worst collection score grade per customer.

2. Build incremental load logic with date watermark filtering directly on `Rcd_DT` (for `silver_customer_cif`), `APPL_DT` (for `silver_ep_underwriting`), and `Appl_DT` (for `silver_cc_underwriting`)—keeping the exact column names `Rcd_DT`, `APPL_DT`, and `Appl_DT` in both the SELECT output and the incremental WHERE filter (do not use `_RAW` alias suffixes)—to reduce compute footprint during recurring pipeline executions.

3. Add Dataform assertion tests for unique primary keys and non-null constraints across incremental silver tables to safeguard data integrity. Add defaultAssertionDataset and defaultDataset to workflow_settings.yaml with valid values like dataform_assertions and dataform.

4. In `acsm_gold`, create an executive Customer 360 table `acsm_gold.gold_aeon_customer360_profile` by joining `silver_customer_cif` with customer-level summaries from `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`, and active credit card usage from `acsm_bronze.dimProduct`.

5. In `acsm_silver`, train a BigQuery ML logistic regression model named `model_delinquency_propensity` as a Dataform operation with output enabled. Train the model on `acsm_gold.gold_aeon_customer360_profile` using customer age (`N_Age`), net income (`B_NetIncome`), annual income (`B_AnnualIncome`), state (`State`), region (`Region`), and occupation (`Occupation`) to predict a binary delinquency risk label (`delinquency_risk_flag`, set to 1 if `combined_unpaid_osp` is greater than 0, and 0 otherwise).
```

> 💡 **Follow-Up Prompt if Dataform DAG Reports Missing `defaultDataset` / `defaultAssertionDataset` in `workflow_settings.yaml`**:
> If the Data Engineering Agent generates assertion nodes without updating `workflow_settings.yaml`, paste this follow-up prompt into the Data Engineering Agent chat:
> ```text
> Add defaultAssertionDataset and defaultDataset to workflow_settings.yaml with valid values like dataform_assertions and dataform.
> ```

#### **Phase 2 Consolidated Prompt — Batch BQML Delinquency Propensity Prediction (`acsm_gold.gold_aeon360_batch_ml_predictions` Predicting Customer Default Risk Flag & Probability Across All 100K Customers) + Automated SLA Scheduling (Paste After Clicking `Run` on Phase 1)**
> 🎯 **What Phase 2 Predicts**: Executes in-database batch inference (`ML.PREDICT`) using `acsm_silver.model_delinquency_propensity` over all **100,000 customers** in `acsm_gold.gold_aeon_customer360_profile` to predict:
> - **`predicted_delinquency_risk_flag`**: Binary delinquency prediction (`1` = High Risk of Unpaid Collection Balance / Default, `0` = Performing).
> - **`predicted_delinquency_risk_flag_probs`**: Predicted delinquency propensity probability (`0.0000` to `1.0000`) alongside each customer's Easy Payment financing exposure, Credit Card limit, CTOS score, and collection grade.

```text
In `acsm_gold`, create a batch ML prediction table named `acsm_gold.gold_aeon360_batch_ml_predictions` by running BigQuery ML batch inference (`ML.PREDICT`) with the trained logistic regression model `acsm_silver.model_delinquency_propensity` over `acsm_gold.gold_aeon_customer360_profile` to predict each customer's delinquency risk flag (`predicted_delinquency_risk_flag`: 1 for high risk of unpaid collection balance, 0 for performing) and delinquency propensity probability (`predicted_delinquency_risk_flag_probs`) alongside their customer profile, Easy Payment financing exposure, Credit Card limit, and collection status. Also set up scheduled workflow execution with Dataplex Data Quality monitoring on assertions to trigger automated alerting on SLA breaches.
```

---
### ▶️ Part C: Review the Visual DAG & Run the Pipeline
1. The **Data Engineering Agent** will generate and refine your visual **Dataform Medallion + BQML DAG** on the Pipeline Canvas:
   - **4 Silver transformation nodes** (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`) with incremental logic & Dataform assertion checks (`uniqueKey`, `nonNull`).
   - **1 Gold Customer 360 join node** (`gold_aeon_customer360_profile`) waiting for all 4 Silver tables + `dimProduct`.
   - **1 Silver BQML Model Training node** (`model_delinquency_propensity`) training the Logistic Regression model over `gold_aeon_customer360_profile`.
   - **1 Gold Batch BQML Scoring node** (`gold_aeon360_batch_ml_predictions`) running `ML.PREDICT` over `gold_aeon_customer360_profile`.
2. Click any node on the canvas to inspect the generated SQL/SQLX definition and assertion rules.
3. Click **`Apply`** to accept the agent's generated nodes, then click **`Run`** at the top of the Pipeline Canvas (or run **Step 5** below as an optional fallback).

---
## Step 5 (Optional Fallback): Invoke `02_medallion_and_reconciliation.sql` if Dataform Pipeline in Step 4 Was Not Run
> **ℹ️ Optional Fallback Step**: If you already executed the **BigQuery Data Engineering Agent (Dataform Pipeline)** in **Step 4**, you can skip this cell and go directly to **Step 6**.
>
> If you were not able to run Dataform in Step 4, run the cell below to invoke [`track1_platform_governance/sql/02_medallion_and_reconciliation.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/02_medallion_and_reconciliation.sql), which builds all **4 Silver tables**, **`acsm_gold.gold_aeon_customer360_profile`**, the BQML model **`acsm_silver.model_delinquency_propensity`**, and **`acsm_gold.gold_aeon360_batch_ml_predictions`**.

In [ ]:
# @title 5.1 (Optional) Invoke `02_medallion_and_reconciliation.sql` (Run Only if Dataform Was Not Executed in Step 4)
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/02_medallion_and_reconciliation.sql
print("✅ Executed 02_medallion_and_reconciliation.sql (Silver tables, Gold Customer 360, BQML Model & Batch Predictions ready)!")

---
## Step 6: Verify `acsm_silver` & `acsm_gold` Tables and Dual-Run Financial Reconciliation (`%%bigquery`)
Run the two `%%bigquery` SQL cells below to:
1. Inspect the created tables and row counts in `acsm_silver` and `acsm_gold`.
2. Run the **Dual-Run Financial Control Total Reconciliation Audit** (`Bronze vs. Silver/Gold`) to prove **0.00 MYR variance** across Easy Payment financed principal (`FIN_AMT`) and Collections unpaid principal (`Unpaid_OSP`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Verify all created Medallion tables across acsm_silver and acsm_gold
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_silver.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
UNION ALL
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_gold.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
ORDER BY medallion_layer, table_name;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 2. Dual-Run Financial Control Total Reconciliation Audit (Bronze vs. Silver) + Preview Gold AEON 360
WITH checks AS (
  SELECT
    'Fact_EP_Judge -> silver_ep_underwriting' AS pipeline_flow,
    'FIN_AMT (Financed Principal MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_ep_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(FIN_AMT AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(FIN_AMT), 2) FROM `acsm_silver.silver_ep_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_CC_Judge -> silver_cc_underwriting' AS pipeline_flow,
    'B_CrLimit (Approved Credit Limit MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_cc_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(B_CrLimit AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(B_CrLimit), 2) FROM `acsm_silver.silver_cc_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_EP_Collection + Fact_CC_Collection -> silver_collections_summary' AS pipeline_flow,
    'Unpaid_OSP (Combined Unpaid Principal MYR)' AS control_metric,
    (SELECT COUNT(DISTINCT CIF_No) FROM (
      SELECT CIF_No FROM `acsm_bronze.Fact_EP_Collection`
      UNION DISTINCT
      SELECT CIF_No FROM `acsm_bronze.Fact_CC_Collection`
    )) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_collections_summary`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Collection`) +
      (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Collection`) AS bronze_total_myr,
    (SELECT ROUND(SUM(combined_unpaid_osp), 2) FROM `acsm_silver.silver_collections_summary`) AS silver_total_myr
)
SELECT
  pipeline_flow,
  control_metric,
  bronze_rows,
  silver_rows,
  bronze_total_myr,
  silver_total_myr,
  (silver_total_myr - bronze_total_myr) AS variance_myr,
  IF(bronze_rows = silver_rows AND ABS(silver_total_myr - bronze_total_myr) = 0, 'PASS (0.00 MYR VARIANCE)', 'INVESTIGATE') AS audit_status
FROM checks;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 3. Preview Top 10 Customers in the Gold Batch BQML Predictions Table (`acsm_gold.gold_aeon360_batch_ml_predictions`)
SELECT
  CIF_ID,
  CIF_NM,
  State,
  B_NetIncome,
  total_ep_financed_myr,
  total_cc_limit_myr,
  combined_unpaid_osp,
  worst_collection_score_grade,
  predicted_delinquency_risk_flag,
  ROUND(predicted_delinquency_risk_flag_probs[OFFSET(0)].prob, 4) AS predicted_delinquency_prob
FROM `acsm_gold.gold_aeon360_batch_ml_predictions`
ORDER BY total_ep_financed_myr + total_cc_limit_myr DESC
LIMIT 10;

---
## Step 7: Apache Airflow (Cloud Composer) Orchestrator to Invoke the Dataform Medallion DAG (`RFP Clauses C1.1.1.5, C1.1.1.6, C1.1.1.7`)
In enterprise production environments, **Cloud Composer (managed Apache Airflow)** acts as the overarching control-plane orchestrator that triggers the **BigQuery Dataform Medallion DAG (`acsm_bronze` $\rightarrow$ `acsm_silver` $\rightarrow$ `acsm_gold`)** after upstream ingestion completes.

The production Airflow DAG is stored in [`track1_platform_governance/airflow/acsm_medallion_dataform_orchestrator_dag.py`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/airflow/acsm_medallion_dataform_orchestrator_dag.py) and orchestrates 4 tasks:

| Airflow Task ID | Google Cloud Provider Operator | Purpose in the ACSM Medallion Pipeline |
| :--- | :--- | :--- |
| **`1. verify_bronze_lakehouse_readiness`** | `BigQueryCheckOperator` | Verifies all 8 `acsm_bronze` tables are populated before compiling the DAG. |
| **`2. compile_dataform_medallion_repo`** | `DataformCreateCompilationResultOperator` | Compiles the `.sqlx` workspace in `asia-southeast1`, resolving all `${ref(...)}` dependencies and assertion graphs. |
| **`3. invoke_dataform_medallion_dag`** | `DataformCreateWorkflowInvocationOperator` | Executes the compiled Dataform DAG (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`, PK/Non-Null Assertions, `model_delinquency_propensity`, `gold_aeon_customer360_profile`, and `gold_aeon360_batch_ml_predictions`). |
| **`4. audit_reconciliation_and_dq_sla`** | `BigQueryCheckOperator` | Enforces `0.00 MYR` Bronze-to-Silver financial reconciliation variance and SLA breach alerting (`on_failure_callback`). |

Run the cell below to invoke `track1_platform_governance/airflow/invoke_dataform_medallion_dag.py`, which compiles and triggers your Dataform Medallion workflow in `asia-southeast1`.

In [ ]:
# @title 7.1 Invoke the Dataform Medallion DAG via the Airflow Orchestrator Script (`invoke_dataform_medallion_dag.py`)
!git -C aeon-credit-gcp-workshop pull --quiet 2>/dev/null || true
!python3 aeon-credit-gcp-workshop/track1_platform_governance/airflow/invoke_dataform_medallion_dag.py